# Apnea detection: dilated residual 1D CNN

## Setup

In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
# Hides TensorFlow's C++ INFO and WARNING lines (the converter prints pages of
# them). Must be set BEFORE tensorflow is imported; errors still show.

import json                      # history.json: the per-epoch metrics, readable without Keras
import random                    # Python's own generator, seeded below with the others
from pathlib import Path         # paths that work the same on macOS, Linux and Windows

import matplotlib.pyplot as plt  # training curves
import numpy as np               # arrays: the ECG windows, labels and predictions
import tensorflow as tf          # model, training, TFLite conversion
from sklearn.metrics import roc_auc_score, roc_curve   # exact AUC and ROC for the curves
from tensorflow.keras import layers                    # the building blocks of the network
from tensorflow.keras.models import Model              # the functional-API model class

tf.get_logger().setLevel("ERROR")
# The same for TensorFlow's Python-side messages. "INFO:tensorflow:Assets written".

# ── folders ──────────────────────────────────────────────────────────────────
ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
# The project root, whichever folder the kernel starts in. VS Code starts a
# notebook's kernel in the notebook's own folder (src/), a terminal-launched
# Jupyter usually at the root; both resolve to the same place.

RAW_DIR = ROOT / "data" / "apnea-ecg"      # PhysioNet recordings, as downloaded (WFDB format)
PROCESSED_DIR = ROOT / "data" / "processed"  # one-minute windows as .npy arrays, ready to train on
RESULTS_DIR = ROOT / "results"               # trained model, history, figures, TFLite file
for folder in (RAW_DIR, PROCESSED_DIR, RESULTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
    # parents=True creates data/ too if needed; exist_ok=True makes it a no-op
    # when the folder is already there, so the cell can be re-run freely.

# ── signal constants ─────────────────────────────────────────────────────────
FS = 100                    # Apnea-ECG sampling rate
WINDOW_SAMPLES = 60 * FS    # one expert label per minute
# Apnea-ECG is sampled at 100 Hz and an expert scored every minute as apnea or
# normal. One minute is therefore the natural unit: one window of 6000
# samples = one label = one prediction.

# ── output files ─────────────────────────────────────────────────────────────
MODEL_PATH = RESULTS_DIR / "model.keras"             # the trained Keras model
HISTORY_PATH = RESULTS_DIR / "history.json"          # loss and metrics for every epoch
CURVES_PATH = RESULTS_DIR / "training_curves.png"    # the four-panel figure

# ── reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
# Weight initialisation, dropout masks and the per-epoch shuffle all draw
# random numbers. Seeding all three generators makes a rerun on the same
# machine and library versions produce the same model, so a change in the
# results can be attributed to a change in the code, not to chance.

## Dataset

In [ ]:
import wfdb          # PhysioNet's library: downloads records, reads signals (.dat) and annotations
from scipy.signal import butter, filtfilt, iirnotch, sosfiltfilt   # filter design and application

# ── which records go where ───────────────────────────────────────────────────
# Apnea-ECG has 70 overnight recordings of 7 to 10 hours:
#   a01-a20  20 patients with clear apnea
#   b01-b05   5 borderline cases
#   c01-c10  10 controls, almost no apnea
#   x01-x35  35 recordings released as the official test set
LEARNING_RECORDS = ([f"a{i:02d}" for i in range(1, 21)]
                    + [f"b{i:02d}" for i in range(1, 6)]
                    + [f"c{i:02d}" for i in range(1, 11)])
# f"a{i:02d}" pads to two digits: 1 -> "a01", matching PhysioNet's file names.

VAL_RECORDS = ["a05", "a10", "a15", "a16", "a17", "b04", "c04", "c08"]
# Held out from training to monitor it (early stopping, learning-rate
# schedule). Drawn from all three groups, so validation contains the same mix
# of severe, borderline and normal nights as the data it stands in for.

SPLITS = {
    "train": [r for r in LEARNING_RECORDS if r not in VAL_RECORDS],   # 27 records
    "val": VAL_RECORDS,                                               #  8 records
    "test": [f"x{i:02d}" for i in range(1, 36)],                      # 35 records
}
# The split is BY RECORD, never by minute: every minute of one patient lands
# in the same set. Splitting minutes at random puts the same heart on both
# sides, and the network learns to recognise the patient instead of the
# apnea.

# ── download what is missing ─────────────────────────────────────────────────
missing = [r for records in SPLITS.values() for r in records
           if not all((RAW_DIR / f"{r}{ext}").exists() for ext in (".hea", ".dat", ".apn", ".qrs"))]
# A record is complete only with all four files:
#   .hea  text header: sampling rate, length, scaling of the signal
#   .dat  the samples themselves (about 6 MB per night)
#   .apn  the expert's A / N label for every minute
#   .qrs  the position of every heartbeat (R peak), detected automatically.
#         The model reads the raw waveform and does not use it; it is kept so
#         the local copy is complete for any beat-based analysis later.
if missing:
    print(f"downloading {len(missing)} records to {RAW_DIR} ...")
    wfdb.dl_database("apnea-ecg", dl_dir=str(RAW_DIR), records=missing, annotators=["apn", "qrs"])
    # Only the missing records, with both annotation files. The database also
    # holds respiration recordings (a01r...), which have no ECG channel and
    # are skipped.
else:
    print("raw records: all present")


def clean_ecg(x: np.ndarray) -> np.ndarray:
    """A whole night of raw ECG (mV) -> filtered and clipped, same length."""
    x = sosfiltfilt(butter(4, [0.5, 40], btype="band", fs=FS, output="sos"), x)   # baseline wander, noise
    # Butterworth band-pass keeping 0.5-40 Hz:
    #   below 0.5 Hz  baseline wander, from breathing and body movement
    #   above 40 Hz   muscle activity and electronic noise
    # output="sos" (second-order sections) keeps the filter numerically
    # stable. sosfiltfilt runs it forward then backward, so the two phase
    # shifts cancel and the heartbeats do not move in time. That needs the
    # whole recording at once, which is fine here, offline.

    x = filtfilt(*iirnotch(50, Q=30, fs=FS), x)                                  # mains
    # Narrow notch at 50 Hz, the mains frequency (Europe, Morocco). At 100 Hz
    # sampling, 50 Hz is exactly the highest frequency the signal can hold,
    # and the band-pass above already removes it; the notch is kept so the
    # windows stay identical to the ones the model was trained on.

    x = np.nan_to_num(x, nan=np.nanmedian(x))
    # Safeguard: replaces any missing sample with the night's median. The
    # median, not the mean, because R peaks and artifacts pull the mean.

    return np.clip(x, -5, 5)                                                     # artifacts, mV
    # A real ECG stays well within +-5 mV; beyond that is an artifact (an
    # electrode moving or coming off). Clipping bounds it instead of deleting
    # it, so no sample is removed and the minute grid never shifts.


def record_windows(rec: str):
    """One record -> (list of z-scored 6000-sample windows, list of labels)."""
    signal, fields = wfdb.rdsamp(str(RAW_DIR / rec))
    # signal: (samples, channels) in mV. fields: header info, e.g. "fs".
    assert fields["fs"] == FS, f"{rec}: unexpected fs {fields['fs']}"
    # A record at another rate would make every "minute" the wrong length
    # without any error. Stop instead.

    ann = wfdb.rdann(str(RAW_DIR / rec), "apn")
    # ann.sample: where each labelled minute starts (0, 6000, 12000 ...)
    # ann.symbol: its label, "A" (apnea) or "N" (normal)
    ecg = clean_ecg(signal[:, 0])    # channel 0 is the single ECG lead

    X, y = [], []
    for start, label in zip(ann.sample, ann.symbol):
        if start + WINDOW_SAMPLES > len(ecg) or label not in ("A", "N"):
            continue
            # The only two reasons a minute is ever dropped, both structural:
            # the recording ends before the minute does, or the label is
            # neither A nor N. Never for signal quality: a real device has to
            # answer for the noisy minutes too.
        w = ecg[start:start + WINDOW_SAMPLES].astype(np.float32)
        X.append(np.nan_to_num((w - w.mean()) / (w.std() + 1e-6)))
        # Z-score: subtract the minute's mean, divide by its standard
        # deviation. The +1e-6 avoids a division by zero on a flat minute.
        y.append(label == "A")   # True = apnea, stored below as 1.0
    return X, y
    # Each minute z-scored on its own: absolute amplitude reflects electrode
    # placement, not breathing. Labels are float32 (see np.asarray below):
    # Keras compares them with the float predictions.


# ── build the three splits ───────────────────────────────────────────────────
for split, records in SPLITS.items():
    x_path = PROCESSED_DIR / f"X_{split}_apnea.npy"
    y_path = PROCESSED_DIR / f"y_{split}_apnea.npy"
    if x_path.exists() and y_path.exists():
        print(f"{split:5s}: already processed")
        continue
        # Once built, a split is skipped. Delete data/processed/ to force a rebuild.
    X, y = [], []
    for rec in records:
        X_rec, y_rec = record_windows(rec)
        X += X_rec
        y += y_rec
    np.save(x_path, np.stack(X))
    # (minutes, 6000) float32. No channel axis yet: it is added when loading
    # (X[..., np.newaxis]), because Conv1D wants (minutes, 6000, 1).
    np.save(y_path, np.asarray(y, dtype=np.float32))
    print(f"{split:5s}: {len(y)} minutes from {len(records)} records, {np.mean(y):.1%} apnea")

## Hyperparameters

In [ ]:
# ── stem: shrink the minute before the expensive part ────────────────────────
STEM = (
    #  filt  ker  str  pool
    (16,     15,   2,   1),
    (24,      7,   1,   2),
    (40,      7,   1,   2),
    (64,      5,   1,   2),
)
# Four convolution layers, one row each:
#   filt  number of filters (feature channels) the layer produces
#   ker   kernel length in samples: how much signal each filter looks at
#   str   stride: 2 keeps every second output, halving the length
#   pool  max-pooling factor after the layer (1 = none)
# Length through the stem: 6000 -> 3000 -> 1500 -> 750 -> 375 timesteps,
# one step every 16 samples (160 ms). The first kernel, 15 samples
# = 150 ms, is about one QRS complex wide, so the first layer can already see
# a whole heartbeat. The last row's 64 filters are replaced by TRUNK_FILTERS
# in build_model(), so the stem hands the trunk exactly the width it uses.

# ── trunk: residual blocks that look far back in time ────────────────────────
TRUNK_FILTERS = 32                    # channels in every residual block
TRUNK_KERNEL = 3                      # kernel length of the dilated convolutions
DILATIONS = (1, 2, 4, 8, 16, 32, 64)  # one block per value
# A dilated convolution with dilation d reads every d-th step. Doubling d at
# each block makes the view grow exponentially while the parameter count
# grows only linearly: after seven blocks one output sees 8225 input samples
# (82 s), more than the whole 60 s minute. build_model() checks this.
SE_RATIO = 8                          # squeeze-excite bottleneck: 32 channels -> 4 -> 32
DROPOUT_TRUNK = 0.10                  # share of channels switched off per block, training only
DROPOUT_HEAD = 0.30                   # share of units switched off before the output
# Dropout fights overfitting: with only 27 training patients the network can
# memorise individuals. It is active during training only; at inference
# every unit is used.

# ── training ─────────────────────────────────────────────────────────────────
EPOCHS = 70               # upper bound; early stopping usually ends the run far sooner
BATCH_SIZE = 64           # minutes per gradient step
LEARNING_RATE = 1e-3      # Adam's usual starting step size
EARLY_STOP_PATIENCE = 12  # epochs without a better val_auc before training stops
REDUCE_LR_PATIENCE = 6    # epochs without a better val_auc before the learning rate is halved
# REDUCE_LR_PATIENCE < EARLY_STOP_PATIENCE on purpose: when validation stalls,
# the run first gets a smaller step size and a second chance to improve
# before it is stopped.

## Data

In [ ]:
def load_split(split: str):
    """One split from data/processed -> (X of shape (minutes, 6000, 1), y of shape (minutes,))."""
    X = np.load(PROCESSED_DIR / f"X_{split}_apnea.npy")
    y = np.load(PROCESSED_DIR / f"y_{split}_apnea.npy")
    assert len(X) == len(y), f"{split}: X and y disagree"
    # One label per window. A mismatch means the two files come from
    # different preprocessing runs: stop rather than train on shifted labels.
    return X[..., np.newaxis], y
    # [..., np.newaxis] adds the channel axis: (minutes, 6000) becomes
    # (minutes, 6000, 1). Conv1D expects (batch, timesteps, channels), and a
    # single ECG lead is one channel.


X_tr, y_tr = load_split("train")   # 27 patients: the network learns from these
X_val, y_val = load_split("val")   #  8 patients: never trained on, used to judge each epoch
# The test split (the 35 x records) is not loaded: it is kept for the final
# evaluation, once every training decision has been made.
print(f"train {X_tr.shape}  apnea {y_tr.mean():.1%}")
print(f"val   {X_val.shape}  apnea {y_val.mean():.1%}")
# The mean of a 0/1 label array is the share of apnea minutes.

## Building Blocks

In [ ]:
def receptive_field() -> tuple[int, int]:
    """How many input samples one trunk output can see, and how many samples one trunk step spans."""
    span, stride = 1, 1
    # span:   width of the input region seen so far, in samples
    # stride: distance in input samples between two neighbouring outputs
    for _filters, kernel, conv_stride, pool in STEM:
        span += (kernel - 1) * stride      # a kernel of k widens the view by k - 1 steps
        stride *= conv_stride              # a strided layer spaces later steps further apart
        if pool > 1:
            span += (pool - 1) * stride    # pooling widens the view like a kernel of size pool
            stride *= pool
    for dilation in DILATIONS:
        effective_kernel = 1 + (TRUNK_KERNEL - 1) * dilation   # kernel 3 at dilation d spans 2d + 1 steps
        span += 2 * (effective_kernel - 1) * stride            # two convolutions per residual block
    return span, stride


def squeeze_excite(x, ratio: int, name: str):
    """Channel attention: learn how much each feature channel matters for THIS minute."""
    filters = x.shape[-1]
    se = layers.GlobalAveragePooling1D(name=f"{name}_se_squeeze")(x)
    # "Squeeze": average each channel over the whole minute -> one number per channel.
    se = layers.Dense(max(filters // ratio, 4), activation="relu", name=f"{name}_se_reduce")(se)
    # Bottleneck (32 -> 4): forces a compact summary of which channels are active.
    se = layers.Dense(filters, activation="sigmoid", name=f"{name}_se_expand")(se)
    # "Excite": back to one weight per channel, each between 0 and 1.
    se = layers.Reshape((1, filters), name=f"{name}_se_reshape")(se)
    # (batch, channels) -> (batch, 1, channels), so it broadcasts over every timestep.
    return layers.Multiply(name=f"{name}_se_scale")([x, se])
    # Every channel is scaled by its weight: useful features are kept,
    # irrelevant ones turned down, differently for every minute.


def conv_bn(x, filters: int, kernel: int, conv_name: str, bn_name: str,
            dilation: int = 1, strides: int = 1):
    """Convolution followed by batch normalisation: the pair used everywhere below."""
    x = layers.Conv1D(filters, kernel, strides=strides, padding="same", dilation_rate=dilation,
                      use_bias=False, name=conv_name)(x)
    # padding="same" keeps the length unchanged (unless strides > 1), so the
    # shortcut and the main path of a residual block can be added.
    # use_bias=False: the batch normalisation right after has its own shift
    # term, so a bias here would be a redundant parameter.
    return layers.BatchNormalization(name=bn_name)(x)
    # Re-centres and rescales each channel over the batch. It keeps the
    # activations in a stable range, which makes deep networks train faster
    # and less sensitively to the learning rate. On the device it is folded
    # into the convolution's weights and costs nothing.


def residual_block(x, filters: int, kernel: int, dilation: int, dropout: float, name: str):
    """Two dilated convolutions plus a shortcut: output = relu(input + learned correction)."""
    shortcut = x
    if x.shape[-1] != filters:
        shortcut = conv_bn(x, filters, 1, f"{name}_proj", f"{name}_proj_bn")
        # A 1x1 convolution only when the channel counts differ, so the two
        # paths can be added. Here the stem already outputs TRUNK_FILTERS
        # channels, so every shortcut is the plain identity.

    y = conv_bn(x, filters, kernel, f"{name}_conv1", f"{name}_bn1", dilation)
    y = layers.Activation("relu", name=f"{name}_relu1")(y)
    y = layers.SpatialDropout1D(dropout, name=f"{name}_drop")(y)
    # SpatialDropout1D switches off whole CHANNELS, not single samples.
    # Neighbouring ECG samples are so correlated that dropping one sample
    # hides nothing; dropping a whole channel forces the other channels to
    # carry the information too.
    y = conv_bn(y, filters, kernel, f"{name}_conv2", f"{name}_bn2", dilation)
    y = squeeze_excite(y, SE_RATIO, name)

    y = layers.Add(name=f"{name}_add")([shortcut, y])
    # The shortcut: the block only has to learn a correction to its input.
    # It also gives gradients a direct path back through all seven blocks, so
    # the early layers still learn.
    return layers.Activation("relu", name=f"{name}_out")(y)


def attention_pool(x, name: str = "attn"):
    """Weighted average over time, with the weights learned: where in the minute to look."""
    scores = layers.Conv1D(1, 1, name=f"{name}_score")(x)
    # One score per timestep: how relevant this moment of the minute looks.
    weights = layers.Softmax(axis=1, name=f"{name}_softmax")(scores)
    # Softmax over time: positive weights that sum to 1 across the 375 steps.
    pooled = layers.Dot(axes=1, name=f"{name}_pool")([x, weights])
    # Weighted sum over time: (batch, 375, 32) x (batch, 375, 1) -> (batch, 32, 1).
    return layers.Flatten(name=f"{name}_flat")(pooled)
    # An apnea shows up in a few seconds of the minute, typically the heart
    # rate swing when breathing restarts. A plain average dilutes those
    # seconds into the other fifty; attention lets the model weight them up.

## Model

In [ ]:
def build_model(input_length: int = WINDOW_SAMPLES) -> Model:
    """ECG minute (6000, 1) -> stem -> 7 residual blocks -> 3 poolings -> P(apnea)."""
    span, _stride = receptive_field()
    assert span >= input_length, (
        f"receptive field {span} samples ({span / FS:.2f} s) does not cover "
        f"the {input_length}-sample window")
    # If a change to STEM or DILATIONS leaves the view shorter than the
    # minute, the model could not relate its start to its end. Fail here,
    # loudly, instead of training a model that is blind to part of its input.

    inputs = layers.Input(shape=(input_length, 1), name="ecg")
    # The batch dimension is left free; Keras adds it.

    # ── stem ─────────────────────────────────────────────────────────────────
    x = inputs
    for i, (filters, kernel, conv_stride, pool) in enumerate(STEM, start=1):
        if i == len(STEM):
            filters = TRUNK_FILTERS
            # The last stem layer outputs the trunk's width directly, so no
            # projection is needed in the first residual block.
        x = conv_bn(x, filters, kernel, f"stem{i}_conv", f"stem{i}_bn", strides=conv_stride)
        x = layers.Activation("relu", name=f"stem{i}_relu")(x)
        if pool > 1:
            x = layers.MaxPooling1D(pool, name=f"stem{i}_pool")(x)
            # Keeps the strongest response in each group of `pool` steps and
            # halves the length: cheaper layers, and tolerance to small shifts.

    # ── trunk ────────────────────────────────────────────────────────────────
    for i, dilation in enumerate(DILATIONS, start=1):
        x = residual_block(x, TRUNK_FILTERS, TRUNK_KERNEL, dilation, DROPOUT_TRUNK,
                           name=f"block{i}_d{dilation}")
    # x is now (batch, 375, 32): 375 moments of the minute, 32 features each.

    # ── pooling: from a sequence to one vector per minute ───────────────────
    pooled = layers.Concatenate(name="pool_concat")([
        attention_pool(x),                                   # the moments the model finds relevant
        layers.GlobalAveragePooling1D(name="avg_pool")(x),   # the overall level across the minute
        layers.GlobalMaxPooling1D(name="max_pool")(x),       # the single strongest event
    ])
    # Three complementary summaries side by side: 3 x 32 = 96 numbers.

    # ── head: the decision ───────────────────────────────────────────────────
    y = layers.Dense(64, activation="relu", name="head_dense")(pooled)
    y = layers.Dropout(DROPOUT_HEAD, name="head_drop")(y)
    outputs = layers.Dense(1, activation="sigmoid", name="apnea_prob")(y)
    # One unit with a sigmoid: a number between 0 and 1, read as the
    # probability that this minute is apnea.

    return Model(inputs, outputs, name="apnea_dilated_cnn")


model = build_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    # Adam adapts the step size of every weight to its own gradient history;
    # a robust default that needs little tuning.
    loss=tf.keras.losses.BinaryCrossentropy(),
    # The standard loss for a yes/no output. It punishes confident mistakes
    # hardest: predicting 0.99 for a normal minute costs far more than 0.6.
    metrics=["accuracy",
             tf.keras.metrics.AUC(name="auc"),
             tf.keras.metrics.Recall(name="recall"),
             tf.keras.metrics.Precision(name="precision")],
    # Reported every epoch, not optimised directly:
    #   accuracy   share of minutes classified right, at a 0.5 cut-off
    #   auc        how well apnea minutes are ranked above normal ones, at
    #              every cut-off at once: the metric the callbacks watch
    #   recall     share of apnea minutes caught (sensitivity), at 0.5
    #   precision  share of "apnea" answers that were right, at 0.5
)
# model.summary()

## Training

In [ ]:
class LearningRateLog(tf.keras.callbacks.Callback):
    """Writes the current learning rate into the history at the end of every epoch."""
    def on_epoch_end(self, epoch, logs=None):
        if logs is not None:
            logs["lr"] = float(tf.keras.backend.get_value(self.model.optimizer.learning_rate))
    # Keras does not record the learning rate by itself. With it in
    # history.json, the effect of ReduceLROnPlateau can be read back later.


train_ds = (tf.data.Dataset.from_tensor_slices((X_tr, y_tr))
            .shuffle(len(y_tr), seed=SEED, reshuffle_each_iteration=True)
            .batch(BATCH_SIZE)
            .prefetch(tf.data.AUTOTUNE))
# from_tensor_slices: one element per minute, (window, label).
# shuffle: the buffer holds the WHOLE training set, so the order is a true
#   uniform shuffle, not a sliding window that would keep minutes of the same
#   night together. reshuffle_each_iteration gives a new order every epoch,
#   so batches never repeat.
# batch: groups 64 minutes into one gradient step.
# prefetch: prepares the next batch while the current one trains.

history = model.fit(
    train_ds,
    validation_data=(X_val, y_val),
    # After every epoch the model is scored on the 8 validation patients. It
    # never trains on them, so val_* shows how it does on unseen people.
    epochs=EPOCHS,
    shuffle=False,
    # The dataset above already shuffles; Keras' own shuffle does not apply to
    # a tf.data pipeline and would only print a warning.
    callbacks=[
        LearningRateLog(),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_auc", mode="max", patience=EARLY_STOP_PATIENCE,
            restore_best_weights=True, verbose=1),
        # Stops once val_auc has not improved for 12 epochs, and puts back the
        # weights of the BEST epoch, not the last one. The epochs after the
        # best are the model starting to memorise the training patients.
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_auc", mode="max", factor=0.5,
            patience=REDUCE_LR_PATIENCE, min_lr=1e-6, verbose=1),
        # Halves the learning rate after 6 epochs without improvement: smaller
        # steps can settle into a minimum that large steps kept jumping over.
        # Both callbacks watch AUC because it does not depend on a threshold;
        # the 0.5 behind accuracy is not the one used in deployment.
    ],
    verbose=1,   # one progress bar per epoch
)
# history.history: a dict of lists, one value per epoch for every loss and
# metric (train and val_), plus "lr".

## Save

In [ ]:
model.save(MODEL_PATH)
# Architecture and weights in one file. These are the RESTORED weights (best
# val_auc epoch), because EarlyStopping put them back before fit() returned.

record = {k: [float(v) for v in vals] for k, vals in history.history.items()}
HISTORY_PATH.write_text(json.dumps(record, indent=1))
# float(v): the history holds NumPy numbers, which json cannot write. The file
# lets the curves be redrawn, or two runs compared, without retraining.

print(f"saved {MODEL_PATH}")
print(f"saved {HISTORY_PATH}")

## Training curves

In [ ]:
prob_val = model.predict(X_val, verbose=0).ravel()
# P(apnea) for every validation minute, from the restored model. .ravel()
# flattens (minutes, 1) to (minutes,).
epochs = np.arange(1, len(record["loss"]) + 1)      # 1, 2, ... for the x axis
best = int(np.argmax(record["val_auc"])) + 1        # the epoch whose weights were kept

fig, axes = plt.subplots(2, 2, figsize=(10.5, 7))
panels = [
    (axes[0, 0], "Loss", [("Train", "loss"), ("Validation", "val_loss")]),
    (axes[0, 1], f"AUC (best val {max(record['val_auc']):.4f} at epoch {best})",
     [("Train", "auc"), ("Validation", "val_auc")]),
    (axes[1, 0], "Validation precision & recall at 0.5",
     [("Precision", "val_precision"), ("Recall", "val_recall")]),
]
for ax, title, series in panels:
    for label, key in series:
        ax.plot(epochs, record[key], label=label, linewidth=1.8)
    ax.axvline(best, color="grey", linestyle="--", linewidth=1)
    # The dashed line marks the restored epoch.
    ax.set(title=title, xlabel="Epoch")
    ax.grid(alpha=0.3)
    ax.legend()

fpr, tpr, _ = roc_curve(y_val, prob_val)
# The ROC curve: for every possible cut-off, the share of normal minutes
# wrongly flagged (false positive rate) against the share of apnea minutes
# caught (true positive rate).
ax = axes[1, 1]
ax.plot([0, 1], [0, 1], color="grey", linestyle="--", linewidth=1)
# The diagonal is what a coin flip would score.
ax.plot(fpr, tpr, linewidth=1.8, label=f"AUC {roc_auc_score(y_val, prob_val):.4f}")
ax.set(title="Validation ROC", xlabel="False positive rate", ylabel="True positive rate")
ax.set_aspect("equal", adjustable="box")   # square, so the curve's shape is not distorted
ax.grid(alpha=0.3)
ax.legend(loc="lower right")

fig.tight_layout()                  # stops titles and labels from overlapping
fig.savefig(CURVES_PATH, dpi=150)
plt.show()
print(f"saved {CURVES_PATH}")

## TensorFlow Lite conversion

Converts `results/model.keras` into a fully int8 model for TFLite Micro on the ESP32-S3.

- **Fixed input shape `(1, 6000, 1)`**: a microcontroller allocates every tensor up front.
- **Full int8**: weights, activations, input and output. Activation ranges are calibrated on
  500 real training windows.
- The model is exported as a SavedModel first. Converting the Keras 3 object directly leaves the
  weights as variables, and int8 calibration then fails.

Only needs the Setup, Hyperparameters and Data cells; it does not need retraining.

In [ ]:
import tempfile   # a scratch folder that deletes itself

TFLITE_PATH = RESULTS_DIR / "model_int8.tflite"

model = tf.keras.models.load_model(MODEL_PATH)
# Reloaded from disk, so this section works on its own after a kernel
# restart, without retraining.


def representative_dataset():
    """500 real training minutes, one at a time, for the converter to calibrate int8 ranges."""
    rng = np.random.default_rng(SEED)
    for i in rng.choice(len(X_tr), size=500, replace=False):
        yield [X_tr[i:i + 1]]
    # The converter runs these through the float model and records how large
    # every internal value gets. Those ranges set each tensor's int8 scale.
    # They must be REAL windows: random noise would give the wrong ranges and
    # cost accuracy. No labels are needed. X_tr[i:i + 1] keeps the batch
    # axis: shape (1, 6000, 1).


with tempfile.TemporaryDirectory() as saved_model_dir:
    # A temporary folder, deleted with its contents when the block ends: the
    # SavedModel is only an intermediate step.
    model.export(saved_model_dir, format="tf_saved_model", verbose=False,
                 input_signature=[tf.TensorSpec([1, WINDOW_SAMPLES, 1], tf.float32, name="ecg")])
    # SavedModel: the network with its weights frozen as constants. Converting
    # the Keras 3 model object directly leaves the weights as variables, and
    # int8 calibration then fails. The input shape is fixed at one window of
    # 6000 samples: a microcontroller reserves all its memory up front and
    # cannot handle a variable batch size.

    converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]      # turn quantisation on
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    # Only int8 operations are allowed. If any layer could not be quantised,
    # conversion fails instead of silently keeping a float operation the
    # microcontroller would run slowly or not at all.
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    # Input and output in int8 too: no float left anywhere in the model. The
    # device quantises the z-scored window before inference and turns the
    # int8 output back into a probability after.
    tflite_model = converter.convert()   # the model as bytes, in TFLite's flatbuffer format

TFLITE_PATH.write_bytes(tflite_model)
print(f"saved {TFLITE_PATH} ({len(tflite_model) / 1024:.1f} KB)")

## Test set

Scores `results/model.keras` and `results/model_int8.tflite` on the 35 test records, which played no
part in training. For each model the threshold is chosen on the validation records, then applied
once to the test set.

Needs the Setup, Dataset and Data cells; it does not need retraining.

In [ ]:
import warnings

from sklearn.metrics import confusion_matrix

warnings.filterwarnings("ignore", message=".*tf.lite.Interpreter is deprecated.*")
# tf.lite.Interpreter still works; TensorFlow only announces that it will move
# to the separate LiteRT package.

TFLITE_PATH = RESULTS_DIR / "model_int8.tflite"
SMOOTHING = True                               # False: score the raw per-minute outputs
SMOOTH_WINDOW = 5                              # minutes averaged into each decision
THRESHOLD_GRID = np.arange(0.05, 0.96, 0.01)   # candidate thresholds, tried on validation

X_te, y_te = load_split("test")


def record_minutes(rec: str) -> int:
    """How many minutes the Dataset cell kept for one record."""
    n_samples = wfdb.rdheader(str(RAW_DIR / rec)).sig_len
    ann = wfdb.rdann(str(RAW_DIR / rec), "apn")
    return sum(start + WINDOW_SAMPLES <= n_samples and label in ("A", "N")
               for start, label in zip(ann.sample, ann.symbol))
    # The same two rules as record_windows(), applied to the header and the
    # labels only, so no signal is loaded. The arrays hold all records one
    # after another; these counts say where each night ends.


counts = {split: [record_minutes(r) for r in SPLITS[split]] for split in ("val", "test")}
assert sum(counts["val"]) == len(y_val) and sum(counts["test"]) == len(y_te), \
    "record lengths do not match the arrays: delete data/processed and rerun the Dataset cell"


def smooth(prob: np.ndarray, counts: list[int]) -> np.ndarray:
    """Centred moving average over SMOOTH_WINDOW minutes, within each night separately."""
    if not SMOOTHING:
        return prob
        # The probabilities pass through unchanged, so the threshold is chosen,
        # and the test set scored, on each minute's own prediction.
    out, start = np.empty_like(prob), 0
    for n in counts:
        padded = np.pad(prob[start:start + n], SMOOTH_WINDOW // 2, mode="edge")
        out[start:start + n] = np.convolve(padded, np.ones(SMOOTH_WINDOW) / SMOOTH_WINDOW, mode="valid")
        start += n
    return out
    # Apnea comes in runs of several minutes while the model scores each minute
    # alone; averaging neighbours removes isolated spikes. np.pad repeats the
    # edge values so the first and last minutes of a night are averaged over a
    # full window. Night by night, so one patient's night is never averaged
    # with the next patient's.


keras_model = tf.keras.models.load_model(MODEL_PATH)

interpreter = tf.lite.Interpreter(
    model_path=str(TFLITE_PATH),
    experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES)
# Plain built-in kernels: the desktop XNNPACK delegate cannot prepare this int8
# graph, and the reference kernels are closer to what TFLite Micro runs.
interpreter.allocate_tensors()
inp, out = interpreter.get_input_details()[0], interpreter.get_output_details()[0]
in_scale, in_zero = inp["quantization"]
out_scale, out_zero = out["quantization"]
# int8 holds whole numbers from -128 to 127. Each tensor maps them to real
# values with two constants:  real = scale x (q - zero_point).


def predict_keras(X: np.ndarray) -> np.ndarray:
    return keras_model.predict(X, verbose=0).ravel()


def predict_int8(X: np.ndarray) -> np.ndarray:
    """Run the .tflite model on each window in turn, as the device does."""
    probs = np.empty(len(X), dtype=np.float32)
    for i, x in enumerate(X):
        q = np.clip(np.round(x / in_scale) + in_zero, -128, 127).astype(np.int8)
        # Quantise the z-scored window: real value -> int8. np.clip keeps a
        # rare extreme sample inside the int8 range instead of wrapping around.
        interpreter.set_tensor(inp["index"], q[np.newaxis])   # add the batch axis: (1, 6000, 1)
        interpreter.invoke()
        probs[i] = (int(interpreter.get_tensor(out["index"])[0, 0]) - out_zero) * out_scale
        # Dequantise the output: int8 -> probability. int(...) first, so the
        # subtraction cannot overflow the int8 type.
    return probs


def scores(y: np.ndarray, prob: np.ndarray, threshold: float) -> dict:
    tn, fp, fn, tp = confusion_matrix(y.astype(int), (prob > threshold).astype(int), labels=[0, 1]).ravel()
    return {"auc": roc_auc_score(y, prob),
            "accuracy": (tp + tn) / len(y),
            "sensitivity": tp / (tp + fn),               # share of apnea minutes detected
            "specificity": tn / (tn + fp),               # share of normal minutes left alone
            "precision": tp / (tp + fp) if tp + fp else 0.0,
            "tn": tn, "fp": fp, "fn": fn, "tp": tp}


print(f"test set: {len(counts['test'])} records, {len(y_te)} minutes, "
      f"{f'{SMOOTH_WINDOW}-minute smoothing' if SMOOTHING else 'no smoothing'}, threshold chosen on validation\n")
print(f"{'model':6s} {'threshold':>9s} {'AUC':>7s} {'accuracy':>9s} {'sensitivity':>12s} "
      f"{'specificity':>12s} {'precision':>10s}")
results = {}
for name, predict in (("Keras", predict_keras), ("int8", predict_int8)):
    prob_val = smooth(predict(X_val), counts["val"])
    threshold = float(max(THRESHOLD_GRID, key=lambda t: np.mean((prob_val > t) == y_val)))
    # The threshold that maximises accuracy on validation. It is picked there,
    # never on test: a threshold tuned on the test set would make the test
    # score optimistic. Each model gets its own, since quantisation shifts the
    # probabilities slightly.
    prob_te = smooth(predict(X_te), counts["test"])
    results[name] = s = scores(y_te, prob_te, threshold)
    print(f"{name:6s} {threshold:9.2f} {s['auc']:7.4f} {s['accuracy']:9.2%} {s['sensitivity']:12.3f} "
          f"{s['specificity']:12.3f} {s['precision']:10.3f}")

for name, s in results.items():
    print(f"\n{name} confusion matrix      predicted N  predicted A")
    print(f"      true N              {s['tn']:11d}  {s['fp']:11d}")
    print(f"      true A              {s['fn']:11d}  {s['tp']:11d}")